# Defense 7 — Text Input Purification
## Defending against Synonym/Typo Attacks on Text Models

**Concept:** Attackers use tools like TextFooler to swap words with synonyms or introduce slight typos to fool NLP models. A simple but highly effective defense is an input purification pipeline: spell-checking and filtering out rare/adversarial synonyms before the text reaches the model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import textwrap

# We'll simulate a fragile sentiment analysis model
def simulate_sentiment_model(text):
    text = text.lower()
    # Vulnerabilities: The model gets confused by these specific synonyms and typos
    if 'magnificent' in text or 'st3llar' in text:
        return {'Negative': 0.85, 'Positive': 0.15} # Fooled into thinking it's negative
    if 'awful' in text or 't3rrible' in text:
        return {'Negative': 0.10, 'Positive': 0.90} # Fooled into thinking it's positive
        
    # Normal Behavior
    if 'excellent' in text or 'stellar' in text:
        return {'Negative': 0.05, 'Positive': 0.95} 
    if 'bad' in text or 'terrible' in text:
        return {'Negative': 0.95, 'Positive': 0.05} 
        
    return {'Negative': 0.5, 'Positive': 0.5}

examples = [
    {"clean": "The food here is absolutely excellent!", "attacked": "The food here is absolutely magnificent!"},
    {"clean": "A stellar performance by the lead actor.", "attacked": "A st3llar performance by the lead actor."},
    {"clean": "This movie was incredibly bad.", "attacked": "This movie was incredibly awful."},
    {"clean": "What a terrible experience.", "attacked": "What a t3rrible experience."}
]
print('Model setup and examples loaded.')

In [ ]:
def purify_text(text):
    # In a real scenario, this uses a robust spellchecker or a synonym normalizer.
    # For this demo, we normalize adversarial perturbations back to their semantic root.
    adversarial_dictionary = {
        'magnificent': 'excellent',
        'st3llar': 'stellar',
        'awful': 'bad',
        't3rrible': 'terrible'
    }
    words = text.split()
    purified_words = []
    for w in words:
        # Strip basic punctuation for dictionary lookup
        clean_w = w.lower().strip('!.,')
        if clean_w in adversarial_dictionary:
            # Revert the malicious word and try to preserve the original punctuation
            replacement = adversarial_dictionary[clean_w]
            if w.endswith('!'): replacement += '!'
            elif w.endswith('.'): replacement += '.'
            elif w.endswith(','): replacement += ','
            purified_words.append(replacement)
        else:
            purified_words.append(w)
    return ' '.join(purified_words)

print('Purification pipeline ready.')

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
classes = ['Negative', 'Positive']

def plot_bar(ax, preds, text, color, title):
    ax.bar(classes, [preds[c] for c in classes], color=color)
    ax.set_ylim(0, 1.0)
    
    # Wrap text so it fits nicely above the chart
    wrapped_text = "\n".join(textwrap.wrap(text, width=30))
    ax.set_title(f"{title}\n\n{wrapped_text}", fontsize=11, pad=10)
    ax.set_ylabel('Confidence')

for i, ex in enumerate(examples):
    clean_text = ex['clean']
    attacked_text = ex['attacked']
    defended_text = purify_text(attacked_text)
    
    clean_preds = simulate_sentiment_model(clean_text)
    attacked_preds = simulate_sentiment_model(attacked_text)
    defended_preds = simulate_sentiment_model(defended_text)
    
    # True label colors (Green for Positive, Purple for Negative)
    true_is_positive = clean_preds['Positive'] > 0.5
    normal_color = '#27ae60' if true_is_positive else '#8e44ad'
    
    plot_bar(axes[i, 0], clean_preds, clean_text, normal_color, '1. Clean Input')
    plot_bar(axes[i, 1], attacked_preds, attacked_text, '#e74c3c', '2. Attacked Input (Fooled)')
    plot_bar(axes[i, 2], defended_preds, defended_text, '#2980b9', '3. Purified Input (Restored)')

plt.suptitle('Text Defense: Input Purification Pipeline in Action', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('07_text_defense_grid.png', dpi=150, bbox_inches='tight'); plt.show()